In [ ]:
!pip install -q kailash kailash-ml polars torch torchvision matplotlib python-dotenv


# MLFP05 — Exercise 5.1: Vanilla GAN (The Minimax Game)


**WHAT YOU'LL LEARN:**
- The adversarial minimax game: Generator vs Discriminator
- Why GANs work (Nash equilibrium intuition for professionals)
- Build and train an MLP-based GAN on full MNIST (60K images)
- Diagnose training dynamics: when is D "winning" vs healthy balance
- Visualise generated digits, training progression, and loss dynamics
- Apply synthetic data generation for a Singapore insurance company
facing data scarcity under PDPA privacy regulations


**PREREQUISITES: M5/ex_1 (autoencoders — generative model foundations)**

**ESTIMATED TIME: ~45 min**

**DATASET: MNIST — 60,000 real 28x28 grayscale digits**



In [ ]:
from __future__ import annotations

import asyncio
import copy

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from shared.mlfp05.ex_5 import (
    LATENT_DIM,
    OUTPUT_DIR,
    Generator,
    Discriminator,
    init_environment,
    load_mnist,
    setup_engines,
    close_engines,
    plot_image_grid,
    plot_latent_interpolation,
    plot_training_progression,
    plot_loss_curves,
)

## PHASE 1 — THEORY: The Minimax Game
Imagine a counterfeiter (Generator) trying to produce fake banknotes
that fool a detective (Discriminator). The counterfeiter never sees
real banknotes — they only learn from the detective's feedback:
"this one looks fake because the watermark is wrong."
Over time:
- The counterfeiter improves their forgeries
- The detective gets better at spotting fakes
- Eventually they reach a standoff (Nash equilibrium) where the
detective can't tell real from fake — that's a trained GAN.
Mathematically, this is a minimax game:
min_G max_D [ E[log D(x)] + E[log(1 - D(G(z)))] ]
D wants to maximise: score real images high, fake images low
G wants to minimise: make D score fake images high
When D is perfectly confused (outputs 0.5 for everything),
D_loss converges to ln(4) ≈ 1.386.
MODE COLLAPSE: The biggest GAN failure mode. The counterfeiter finds
ONE type of banknote that fools the detective and keeps making only
that one. In MNIST terms: the generator only produces 1s because
they're the simplest digit. We'll measure this with mode coverage.


In [ ]:
print("=" * 70)
print("  PHASE 1 — THEORY: The Adversarial Minimax Game")
print("=" * 70)
print(
    """
  GAN = Generator + Discriminator in a zero-sum game.

  Generator (counterfeiter):
    - Input: random noise z ~ N(0, 1)
    - Output: fake image that should look real
    - Goal: fool the discriminator

  Discriminator (detective):
    - Input: real OR fake image
    - Output: probability that the image is real
    - Goal: correctly classify real vs fake

  Training alternates:
    1. Train D on a batch of real + fake images
    2. Train G to make D output "real" for fake images

  Nash equilibrium: D outputs 0.5 for everything (can't tell the difference)

  KEY RISK — Mode collapse:
    G discovers one "easy" output (e.g., only digit 1) and stops exploring.
    We detect this by checking whether all 10 digit classes are generated.
"""
)


## PHASE 2 — BUILD: Generator + Discriminator


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 2 — BUILD: Generator and Discriminator Networks")
print("=" * 70)

device = init_environment()
X_real, y_real, real_loader = load_mnist(device)
conn, tracker, exp_name, registry = setup_engines()

# Verify architectures
G_test = Generator().to(device)
D_test = Discriminator().to(device)
z_test = torch.randn(4, LATENT_DIM, device=device)

print(f"\n  Generator: {sum(p.numel() for p in G_test.parameters()):,} parameters")
print(f"    Input:  z ~ N(0, 1), dim={LATENT_DIM}")
print(f"    Output: {tuple(G_test(z_test).shape)} image (28x28)")
print(f"\n  Discriminator: {sum(p.numel() for p in D_test.parameters()):,} parameters")
print(f"    Input:  28x28 image")
print(f"    Output: {tuple(D_test(G_test(z_test)).shape)} scalar logit")


### Checkpoint 1 ─────────────────────────────────────────────────────


In [ ]:
assert G_test(z_test).shape == (4, 1, 28, 28), "Generator output shape wrong"
assert D_test(G_test(z_test)).shape == (4, 1), "Discriminator output shape wrong"
del G_test, D_test, z_test
print("\n--- Checkpoint 1 passed --- G and D architectures verified\n")


## PHASE 3 — TRAIN: Vanilla GAN with Binary Cross-Entropy
L_D = -E[log D(x)] - E[log(1 - D(G(z)))]    (discriminator loss)
L_G = -E[log D(G(z))]                         (generator loss — non-saturating)


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 3 — TRAIN: Vanilla GAN (BCEWithLogitsLoss)")
print("=" * 70)

EPOCHS = 15
LR = 2e-4


async def train_vanilla_gan(epochs: int = EPOCHS, lr: float = LR):
    """Train a vanilla GAN with BCE loss, logging to ExperimentTracker."""
    G = Generator().to(device)
    D = Discriminator().to(device)
    opt_g = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    # TODO: Define BCE loss for adversarial training
    # Hint: bce = nn.BCEWithLogitsLoss()
    bce = ____
    g_losses, d_losses = [], []
    epoch_snapshots = {}

    # Capture initial state (random noise output)
    epoch_snapshots[0] = copy.deepcopy(G.state_dict())

    async with tracker.run(experiment_name=exp_name, run_name="vanilla_gan") as ctx:
        await ctx.log_params(
            {
                "architecture": "Vanilla_GAN_MLP",
                "latent_dim": str(LATENT_DIM),
                "lr": str(lr),
                "epochs": str(epochs),
                "batch_size": "128",
                "loss_type": "BCEWithLogitsLoss",
                "optimizer": "Adam(0.5,0.999)",
            }
        )

        for epoch in range(epochs):
            eg, ed = [], []
            for (real_batch,) in real_loader:
                bs = real_batch.size(0)


### Train Discriminator ──────────────────────────────


In [ ]:
                # D sees real images (label=1) and fake images (label=0)
                z = torch.randn(bs, LATENT_DIM, device=device)
                fake = G(z).detach()
                # TODO: D loss = BCE on real (target=1) + BCE on fake (target=0)
                # Hint: loss_d = bce(D(real_batch), torch.ones(bs, 1, device=device))
                #              + bce(D(fake), torch.zeros(bs, 1, device=device))
                loss_d = ____
                opt_d.zero_grad()
                loss_d.backward()
                opt_d.step()


### Train Generator ──────────────────────────────────


In [ ]:
                # G wants D to classify its fakes as real (target=1)
                z = torch.randn(bs, LATENT_DIM, device=device)
                # TODO: G loss = BCE on D(G(z)) with target=1 (fool the discriminator)
                # Hint: loss_g = bce(D(G(z)), torch.ones(bs, 1, device=device))
                loss_g = ____
                opt_g.zero_grad()
                loss_g.backward()
                opt_g.step()

                eg.append(loss_g.item())
                ed.append(loss_d.item())

            avg_g, avg_d = float(np.mean(eg)), float(np.mean(ed))
            g_losses.append(avg_g)
            d_losses.append(avg_d)
            await ctx.log_metrics({"g_loss": avg_g, "d_loss": avg_d}, step=epoch + 1)
            print(
                f"  [Vanilla GAN] epoch {epoch+1:2d}/{epochs}  "
                f"D={avg_d:.3f}  G={avg_g:.3f}"
            )

            # Capture snapshots for progression visualisation
            if (epoch + 1) in {1, 5, 10, 15}:
                epoch_snapshots[epoch + 1] = copy.deepcopy(G.state_dict())

        await ctx.log_metrics(
            {"final_g_loss": g_losses[-1], "final_d_loss": d_losses[-1]}
        )

    return G, g_losses, d_losses, epoch_snapshots


print("\n  Training vanilla GAN on full MNIST (60K images)...")
G_gan, gan_g_losses, gan_d_losses, gan_snapshots = await train_vanilla_gan()


### Checkpoint 2 ─────────────────────────────────────────────────────


In [ ]:
assert len(gan_g_losses) == EPOCHS, f"Expected {EPOCHS} epochs, got {len(gan_g_losses)}"
# INTERPRETATION: In a healthy GAN, D loss hovers around ln(4) ~ 1.386,
# meaning D is about 50% accurate (can't tell real from fake). If D loss
# drops to 0, D has "won" — it perfectly classifies everything — and G
# gets no useful gradient signal (training collapses).
print("\n--- Checkpoint 2 passed --- vanilla GAN trained\n")


## PHASE 4 — VISUALISE: Generated Gallery, Loss Curves, Progression


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 4 — VISUALISE: What Has the Generator Learned?")
print("=" * 70)

# 4A: Generated image gallery (8x8 grid)
print("\n  4A: Gallery of 64 generated digits")
G_gan.eval()
with torch.no_grad():
    z_gallery = torch.randn(64, LATENT_DIM, device=device)
    gallery_images = G_gan(z_gallery)

fig_gallery = plot_image_grid(
    gallery_images,
    title="Vanilla GAN — Generated Digits (8x8 Grid)",
    save_path=str(OUTPUT_DIR / "ex_5_01_vanilla_gallery.png"),
)
plt.show()

# 4B: Training progression (epoch 0 → 1 → 5 → 10 → 15)
print("\n  4B: Training progression — from noise to digits")
G_progression = Generator().to(device)
fig_progression = plot_training_progression(
    G_progression,
    device,
    gan_snapshots,
    title="Vanilla GAN — Training Progression (Epoch 0 → 15)",
    save_path=str(OUTPUT_DIR / "ex_5_01_vanilla_progression.png"),
)
plt.show()
# Reload the final state after progression visualisation
G_gan.load_state_dict(gan_snapshots[max(gan_snapshots.keys())])
G_gan.eval()

# 4C: G vs D loss dynamics
print("\n  4C: Generator vs Discriminator loss curves")
fig_losses = plot_loss_curves(
    gan_g_losses,
    gan_d_losses,
    title="Vanilla GAN — Training Dynamics",
    save_path=str(OUTPUT_DIR / "ex_5_01_vanilla_losses.png"),
)
plt.show()

# 4D: Latent space interpolation
print("\n  4D: Latent space interpolation — smooth transitions between digits")
fig_interp = plot_latent_interpolation(
    G_gan,
    device,
    title="Vanilla GAN — Latent Interpolation",
    save_path=str(OUTPUT_DIR / "ex_5_01_vanilla_interpolation.png"),
)
plt.show()


### Checkpoint 3 ─────────────────────────────────────────────────────


In [ ]:
import os

assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_01_vanilla_gallery.png")
), "Gallery image should exist"
assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_01_vanilla_progression.png")
), "Progression image should exist"
# INTERPRETATION: The gallery shows whether the generator produces
# diverse, recognisable digits. The progression shows HOW it learns:
# epoch 0 is pure noise, early epochs show blurry shapes, later epochs
# show distinct digits. If all generated images look the same, that's
# mode collapse — the generator found one "easy" output.
print("\n--- Checkpoint 3 passed --- vanilla GAN visualisations complete\n")

# Also save training curves with ModelVisualizer (HTML interactive)
from kailash_ml import ModelVisualizer

viz = ModelVisualizer()
fig_html = viz.training_history(
    metrics={"GAN G loss": gan_g_losses, "GAN D loss": gan_d_losses},
    x_label="Epoch",
    y_label="Loss",
)
fig_html.write_html(str(OUTPUT_DIR / "ex_5_01_vanilla_training.html"))
print("  Interactive training curves saved to ex_5_01_vanilla_training.html")


## PHASE 5 — APPLY: Synthetic Data for Singapore Insurance Under PDPA


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 5 — APPLY: Synthetic Data for Insurance (PDPA Compliance)")
print("=" * 70)
print(
    """
  BUSINESS SCENARIO:
  You are a data scientist at a Singapore life insurance company.
  Your team needs to build a fraud detection model, but the Personal
  Data Protection Act (PDPA) restricts how you can use real policyholder
  records for model training. You have only 2,000 real claim records
  (too few for robust ML), and PDPA compliance review takes 6 months
  for new data access requests.

  SOLUTION: Train a GAN on the limited real data to generate synthetic
  policyholder profiles that preserve statistical properties without
  containing any real person's data. The synthetic data supplements
  real data for model training — no PDPA issues because no real
  personal data is used.

  BUSINESS IMPACT:
  - Model training data increased from 2,000 to 20,000+ records
  - No PDPA compliance delay (synthetic data is not personal data)
  - Fraud detection recall improves from 62% to 78%
  - Estimated annual fraud savings: S$4.2M additional recovered claims
"""
)

# Simulate the insurance scenario using MNIST as a proxy:
# Real policyholder "profiles" = real MNIST digits (limited sample)
# Synthetic profiles = GAN-generated digits
# We compare the statistical distributions to validate quality.

print("\n  Simulating the insurance data scenario with MNIST as proxy...")

# Step 1: Take a small "real" sample (simulating limited insurance data)
rng = np.random.default_rng(42)
small_sample_idx = rng.choice(len(X_real), 2000, replace=False)
X_small_real = X_real[small_sample_idx]
y_small_real = y_real[small_sample_idx]

print(f"  'Real' policyholder records: {len(X_small_real)}")

# Step 2: Generate synthetic data to supplement
# TODO: Generate 18,000 synthetic records using the trained generator
# Hint: Set G_gan to eval mode, generate z from N(0,1) with shape (18000, LATENT_DIM),
#       pass through G_gan with torch.no_grad()
G_gan.eval()
with torch.no_grad():
    z_synthetic = ____  # TODO: torch.randn(18000, LATENT_DIM, device=device)
    X_synthetic = ____  # TODO: G_gan(z_synthetic)

print(f"  Synthetic records generated: {len(X_synthetic)}")
print(f"  Combined dataset: {len(X_small_real) + len(X_synthetic)} records")

# Step 3: Compare real vs synthetic pixel distributions
# TODO: Flatten both tensors for distribution comparison
# Hint: X_real_flat = X_small_real.view(-1).cpu().numpy()
#       X_synth_flat = X_synthetic.view(-1).cpu().numpy()
X_real_flat = ____
X_synth_flat = ____

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    "Real vs Synthetic Distribution Comparison\n"
    "(Insurance Policyholder Profile Proxy)",
    fontsize=14,
    fontweight="bold",
)

# TODO: Plot overlapping histograms for feature distribution
# Hint: axes[0].hist(X_real_flat, bins=50, alpha=0.6, label="Real (2K records)", density=True)
#       axes[0].hist(X_synth_flat, bins=50, alpha=0.6, label="Synthetic (18K)", density=True)
____
____
axes[0].set_xlabel("Feature Value", fontsize=12)
axes[0].set_ylabel("Density", fontsize=12)
axes[0].set_title("Feature Distribution Overlap", fontsize=13)
axes[0].legend(fontsize=11)

# TODO: Plot mean feature values per sample
# Hint: real_means = X_small_real.view(len(X_small_real), -1).mean(dim=1).cpu().numpy()
#       synth_means = X_synthetic.view(len(X_synthetic), -1).mean(dim=1).cpu().numpy()
real_means = ____
synth_means = ____
axes[1].hist(real_means, bins=30, alpha=0.6, label="Real", density=True)
axes[1].hist(synth_means, bins=30, alpha=0.6, label="Synthetic", density=True)
axes[1].set_xlabel("Mean Feature Value per Record", fontsize=12)
axes[1].set_ylabel("Density", fontsize=12)
axes[1].set_title("Record-Level Distribution", fontsize=13)
axes[1].legend(fontsize=11)

# TODO: Plot variance per sample
# Hint: real_vars = X_small_real.view(len(X_small_real), -1).var(dim=1).cpu().numpy()
#       synth_vars = X_synthetic.view(len(X_synthetic), -1).var(dim=1).cpu().numpy()
real_vars = ____
synth_vars = ____
axes[2].hist(real_vars, bins=30, alpha=0.6, label="Real", density=True)
axes[2].hist(synth_vars, bins=30, alpha=0.6, label="Synthetic", density=True)
axes[2].set_xlabel("Feature Variance per Record", fontsize=12)
axes[2].set_ylabel("Density", fontsize=12)
axes[2].set_title("Record Diversity", fontsize=13)
axes[2].legend(fontsize=11)

plt.tight_layout()
fig.savefig(
    str(OUTPUT_DIR / "ex_5_01_vanilla_real_vs_synthetic.png"),
    dpi=150,
    bbox_inches="tight",
)
plt.show()
print("  Distribution comparison saved")

# Step 4: Stakeholder-ready summary
print("\n  ┌────────────────────────────────────────────────────────────┐")
print("  │  STAKEHOLDER SUMMARY: Synthetic Data Quality Assessment   │")
print("  ├────────────────────────────────────────────────────────────┤")
print(f"  │  Real records available:       {len(X_small_real):>8,}                  │")
print(f"  │  Synthetic records generated:  {len(X_synthetic):>8,}                  │")
print(
    f"  │  Combined training set:        {len(X_small_real)+len(X_synthetic):>8,}                  │"
)
print(
    f"  │  Data augmentation factor:     {(len(X_small_real)+len(X_synthetic))/len(X_small_real):.1f}x                       │"
)
print("  │                                                            │")
real_mean_val = float(np.mean(real_means))
synth_mean_val = float(np.mean(synth_means))
mean_diff_pct = abs(real_mean_val - synth_mean_val) / (abs(real_mean_val) + 1e-8) * 100
print(f"  │  Mean feature difference:      {mean_diff_pct:>7.1f}%                  │")
print("  │  PDPA compliance:              No personal data used      │")
print("  │  Status:                       READY FOR MODEL TRAINING   │")
print("  └────────────────────────────────────────────────────────────┘")


### Checkpoint 4 ─────────────────────────────────────────────────────


In [ ]:
assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_01_vanilla_real_vs_synthetic.png")
), "Distribution comparison should exist"
assert len(X_synthetic) == 18000, "Should generate 18K synthetic records"
print("\n--- Checkpoint 4 passed --- insurance application complete\n")


## Cleanup


In [ ]:
await close_engines(conn)


## REFLECTION


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
    """
  VANILLA GAN FUNDAMENTALS:
  [x] The minimax game: Generator (counterfeiter) vs Discriminator (detective)
  [x] Binary cross-entropy loss for adversarial training
  [x] Nash equilibrium: D loss converging to ln(4) ~ 1.386
  [x] Mode collapse risk: generator producing only "easy" outputs

  VISUAL INTUITION:
  [x] Generated digit gallery (8x8 grid) — can you read the digits?
  [x] Training progression: noise → blurry shapes → recognisable digits
  [x] G vs D loss curves — healthy balance vs D domination
  [x] Latent interpolation — smooth transitions prove learned manifold

  REAL-WORLD APPLICATION:
  [x] Synthetic data generation for PDPA-compliant model training
  [x] Statistical validation: real vs synthetic distribution comparison
  [x] Business impact quantification: S$4.2M in additional fraud recovery

  KEY INSIGHT:
  Vanilla GANs work but are UNSTABLE. The BCE loss gives zero gradient
  when D perfectly separates real from fake (JS divergence saturates).
  This means training can suddenly collapse with no warning.

  Next: Exercise 5.2 — WGAN-GP solves this instability with
  Wasserstein distance (smooth gradients even when distributions
  don't overlap) and gradient penalty (replaces weight clipping).
"""
)
